# Lezione 11 — Pratica: casi d'uso reali

Nella lezione sul QAOA hai risolto Max-Cut su un grafo ciclo di sei nodi. In questa pratica chiudi il corso applicando lo stesso algoritmo a un grafo diverso e ragioni su un'applicazione reale di ciò che hai imparato.

## Parte A — QAOA su un grafo diverso

Prima di iniziare, installa le librerie necessarie se lavori in Colab:

`!pip install -q cirq tensorflow tensorflow-quantum networkx`

### Traccia

Nel capitolo sul QAOA hai risolto Max-Cut su un grafo ciclo di sei nodi (`networkx.cycle_graph(6)`). Ora applichi lo stesso circuito QAOA a un grafo completo di cinque nodi (`networkx.complete_graph(5)`), in cui ogni nodo è collegato con tutti gli altri. Adatta lo scheletro del codice, addestra il modello e confronta il taglio medio ottenuto con il massimo teorico calcolato per forza bruta.

**Suggerimenti**

- Riutilizza l'Hamiltoniano di mescolamento e l'Hamiltoniano di costo così come vengono costruiti nel capitolo sul QAOA: cambia solo il grafo di ingresso
- Con cinque nodi la forza bruta è ancora rapida: puoi calcolare il taglio massimo reale per confrontarlo
- Lascia `p = 1` come nell'esempio originale, così il circuito e l'addestramento richiedono lo stesso tempo
- I simboli addestrabili restano `gamma` e `beta`, uno per il blocco di costo e uno per quello di mescolamento

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools


def construir_hamiltoniano_mezcla(qubits):
    """Hamiltoniano de mezcla: igual que en el capítulo de QAOA, una suma de X sobre cada qubit."""
    return sum(cirq.PauliString(cirq.X(qubit)) for qubit in qubits)


def construir_hamiltoniano_coste(grafo, qubits):
    """Hamiltoniano de coste: igual que en el capítulo de QAOA, un término Z_i·Z_j por cada arista."""
    hamiltoniano = grafo.number_of_edges() / 2
    for nodo_a, nodo_b in grafo.edges():
        hamiltoniano += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))
    return hamiltoniano


# TODO: crea el grafo completo de cinco nodos con networkx.complete_graph(5) y guárdalo en 'grafo'
grafo = None

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))  # superposición de todas las divisiones posibles

hamiltoniano_mezcla = construir_hamiltoniano_mezcla(qubits)
hamiltoniano_coste = construir_hamiltoniano_coste(grafo, qubits)

p = 1  # número de repeticiones del ciclo coste-mezcla
gamma, beta = sympy.symbols("gamma beta")
circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(operators=[hamiltoniano_coste, hamiltoniano_mezcla],
                                       coefficients=[gamma, beta])

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)
modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

In [ ]:
# TODO: entrena modelo_qaoa con .fit(entradas, np.array([0]), epochs=200) y guarda el resultado en 'historial'
historial = None

plt.plot(historial.history["loss"])
plt.title("QAOA sobre un grafo completo de 5 nodos — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

# Muestreamos el circuito entrenado para obtener divisiones concretas
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)

# TODO: muestrea el circuito con tfq.layers.Sample(), pasando circuito_con_datos, symbol_names=[gamma, beta],
# symbol_values=parametros_entrenados y repetitions=1000; guarda el resultado en 'muestras'
muestras = None

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

# TODO: calcula el corte máximo real por fuerza bruta (igual que en el capítulo de QAOA), probando
# todos los subconjuntos posibles de nodos con itertools.combinations(grafo.nodes(), r) para cada
# tamaño r, y guárdalo en 'mejor_corte'
mejor_corte = None

print("Corte medio obtenido:", np.mean(cortes_obtenidos))
print("Mejor corte posible (fuerza bruta):", mejor_corte)

<details>
<summary>💡 Mostra soluzione</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools


def construir_hamiltoniano_mezcla(qubits):
    """Hamiltoniano di mescolamento: come nel capitolo sul QAOA, una somma di X su ogni qubit."""
    return sum(cirq.PauliString(cirq.X(qubit)) for qubit in qubits)


def construir_hamiltoniano_coste(grafo, qubits):
    """Hamiltoniano di costo: come nel capitolo sul QAOA, un termine Z_i·Z_j per ogni arco."""
    hamiltoniano = grafo.number_of_edges() / 2
    for nodo_a, nodo_b in grafo.edges():
        hamiltoniano += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))
    return hamiltoniano


# Modifica chiave: un grafo completo di 5 nodi invece del ciclo di 6 nodi del capitolo sul QAOA
grafo = nx.complete_graph(5)

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))  # sovrapposizione di tutte le possibili partizioni

hamiltoniano_mezcla = construir_hamiltoniano_mezcla(qubits)
hamiltoniano_coste = construir_hamiltoniano_coste(grafo, qubits)

p = 1  # numero di ripetizioni del ciclo costo-mescolamento
gamma, beta = sympy.symbols("gamma beta")
circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(operators=[hamiltoniano_coste, hamiltoniano_mezcla],
                                       coefficients=[gamma, beta])

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)
modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

# Addestriamo il modello perché il valore atteso del taglio si avvicini al massimo teorico (etichetta fittizia 0)
historial = modelo_qaoa.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA su un grafo completo di 5 nodi — perdita durante l'addestramento")
plt.xlabel("Iterazione")
plt.ylabel("Perdita")
plt.show()

# Campioniamo il circuito addestrato per ottenere partizioni concrete
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(circuito_con_datos, symbol_names=[gamma, beta],
                                symbol_values=parametros_entrenados, repetitions=1000)

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

# Taglio massimo reale per forza bruta: in un grafo completo di 5 nodi ci sono pochi sottoinsiemi,
# quindi provarli tutti resta rapido
mejor_corte = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

print("Taglio medio ottenuto:", np.mean(cortes_obtenidos))
print("Miglior taglio possibile (forza bruta):", mejor_corte)

# In un grafo completo tutti i nodi sono collegati tra loro, quindi il taglio massimo
# si ottiene dividendo i nodi in due gruppi di dimensioni il più possibile simili (con 5 nodi,
# un gruppo da 2 e uno da 3). Il QAOA con p=1 di solito si avvicina a quel massimo, ma raramente
# lo raggiunge in modo perfetto, come accadeva con il ciclo di 6 nodi del capitolo sul QAOA.
```

</details>

## Parte B — Riflessione finale: casi d'uso reali

Scegli una delle quattro applicazioni viste nella lezione teorica: chimica quantistica (VQE), ottimizzazione combinatoria (QAOA), apprendimento per rinforzo quantistico (QRL) o validazione dell'hardware (supremazia quantistica). Cerca un articolo pubblicato negli ultimi due anni che la applichi a un problema nuovo, diverso dagli esempi di questo corso.

Scrivi un riassunto di tre-cinque frasi che spieghi:

- Quale problema risolve l'articolo
- Quale algoritmo quantistico tra quelli appresi nel corso usa (o una sua variante)
- Quale limitazione menziona l'articolo stesso sullo stato attuale della tecnologia

**Cerca l'articolo e scrivi il tuo riassunto prima di continuare.**

Scrivi qui la tua risposta: